# Stage 6 Static GraphSAGE vs Temporal GraphSAGE
Smoke first, then six-cell validation. Final test is intentionally a separate, gated command.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO_URL = 'REPLACE_WITH_REPOSITORY_URL'
EXPECTED_COMMIT = 'REPLACE_WITH_GNN_COMMIT'
DATASET_ROOT = '/content/drive/MyDrive/DSS/medajob-temporal-v0.3'
OUTPUT_ROOT = '/content/drive/MyDrive/DSS/stage6_gnn'
EXPECTED_MANIFEST_SHA256 = 'a6d983e754f49a9e2076022d48109516a931def7c17b1a725394245872aad287'
assert not REPO_URL.startswith('REPLACE_')
assert not EXPECTED_COMMIT.startswith('REPLACE_')


In [ ]:
import hashlib, os, pathlib, subprocess, sys
repo = pathlib.Path('/content/Temporal-GraphSAGE-DSS')
if not repo.exists(): subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=repo, check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=repo, check=True)
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
assert actual_commit == EXPECTED_COMMIT
manifest = pathlib.Path(DATASET_ROOT) / 'data/candidate55/v0_3/dataset_manifest.json'
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256
pathlib.Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
os.chdir(repo)


In [ ]:
!python -m pip install -r requirements-gnn-colab.txt
!python -m pip install --no-deps -e .
import torch
assert torch.cuda.is_available(), 'CUDA is not available; stop before training'
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
assert gpu.returncode == 0, gpu.stderr
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
print(gpu.stdout[:2000])


In [ ]:
tests = subprocess.run([sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True)
(pathlib.Path(OUTPUT_ROOT) / 'pre_training_tests.log').write_text(f'exit_code={tests.returncode}\n{tests.stdout}{tests.stderr}')
print(tests.stdout)
tests.check_returncode()


In [ ]:
smoke_command = [sys.executable, 'scripts/run_stage6_gnn.py', '--mode', 'smoke', '--device', 'cuda', '--dataset-root', DATASET_ROOT, '--output-dir', f'{OUTPUT_ROOT}/smoke', '--resume']
subprocess.run(smoke_command, check=True)


In [ ]:
# Run only after inspecting smoke/run_receipt.json and both model manifests.
validation_command = [sys.executable, 'scripts/run_stage6_gnn.py', '--mode', 'validation', '--device', 'cuda', '--dataset-root', DATASET_ROOT, '--output-dir', f'{OUTPUT_ROOT}/validation', '--model-seed', '42', '--half-life-days', '7', '14', '30', '--resume']
subprocess.run(validation_command, check=True)


## Final test gate
Do not run final here automatically. Use the guide only after six-cell validation writes and reviews `gnn_config_lock.json`.